# Anharmonic Oscillator: NN Thimble — Lambda Scan (General Endpoints)

This notebook trains a fresh **nonlinear thimble** model
$x = A^\top z + b + g_\theta(z)$
for each $\lambda$ in a configurable list, with **general endpoints** $(x_i, x_f)$.

The bias $b$ is initialised as a linear interpolation between $x_i$ and $x_f$,
which is essential for non-zero endpoints.

For every $\lambda$ it runs **3 different random seeds**, records all results,
and selects the **best seed** (lowest final variance loss).

For every $\lambda$ × seed it records:
- **ML propagator** $K(x_f,T;\,x_i,0)$ with MC std error
- **Basis truncation** independent verification
- **Mehler kernel** (exact, for $\lambda=0$ sanity)
- Loss history, ESS/N
- Arrow (before/after) diagrams

All results go to a single CSV, with a `best` flag marking the winning seed.
A final comparison plot shows ML (best seed) vs basis truncation across $\lambda$.

In [ ]:
import torch, torch.nn as nn, numpy as np, time, csv, os, math
import matplotlib.pyplot as plt
%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)

## 1. Parameters

Set `x_i_prop` and `x_f_prop` to any desired endpoints.
The bias is automatically initialised as a linear interpolation between them.
Change `lambda_list` to control the scan.

In [ ]:
N       = 8 #  16   18th April 2026
T       = 1.0
m       = 1.0
omega   = 2   # 2.0  April 15th 
epsilon = 0.01 # 0.0001   # 0.01 April 15th and all previous runs
D       = N - 1 
rank    = 8   
epochs  = 40000    ## 15000 apil 17th
batch   = 2048 
N_mc    = 100_000

# ── endpoints ──
x_i_prop = 0
x_f_prop = 0
 
#lambda_list = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 2, 3, 4,  8]
 
lambda_list = [1,2,4]

#lambda_list = [0.05, 0.1, 0.15, 0.2, 0.4, 0.8, 0.9, 1.0, 2, 3, 4,  8]
 

print(f"N={N}, T={T}, m={m}, omega={omega}, eps={epsilon}")
print(f"D={D}, rank={rank}, epochs={epochs}, batch={batch}")
print(f"Endpoints: x_i={x_i_prop}, x_f={x_f_prop}")
print(f"Lambda values to scan: {lambda_list}")

## 2. Action Function

The **action** $S[x]$ scores each discretised path and is the physics "cost function" that appears in the path integral $\int \mathcal{D}x \, e^{iS[x]}$. For the anharmonic oscillator with general endpoints $(x_i, x_f)$ and $N$ time slices of size $a = T/N$:

$$S[x] \;=\; \sum_{k=0}^{N-1} \Big[\; \tfrac{m}{2a}(x_{k+1}-x_k)^{2} \;-\; \tfrac{a\,m}{2}(\omega^{2}-i\epsilon)\,x_{k}^{2} \;-\; \tfrac{a\lambda}{4}\,x_{k}^{4}\;\Big]$$

- **Kinetic term** $\tfrac{m}{2a}(x_{k+1}-x_k)^2$ — penalises fast wiggles between time slices.
- **Harmonic potential** $\tfrac{a m \omega^2}{2} x_k^2$ — a spring that pulls the particle back toward the origin.
- **Quartic anharmonicity** $\tfrac{a \lambda}{4} x_k^4$ — the new physics that makes the action **non-Gaussian**. There is no closed-form propagator when $\lambda \neq 0$.
- **$i\epsilon$ regulator** — $\omega^2 \to \omega^2 - i\epsilon$ damps high-frequency modes (Feynman prescription). We use $\epsilon = 0$ here; $\epsilon > 0$ would add a slight Euclidean rotation.

Endpoints are **pinned** ($x_0 = x_i$, $x_N = x_f$, padded inside `complex_action`); only the $D = N-1$ interior points are free. Each path contributes an "arrow" $e^{iS}$ to the path integral — on the real axis these arrows point in random directions and cancel massively (**the sign problem**). The neural network in the next cell will learn a complex contour where they align.

In [ ]:
def complex_action(x, x_i, x_f, lam_val, a_dt, m_val, omega_val, eps_val):
    bsz = x.shape[0]
    pad_i = torch.full((bsz, 1), x_i, dtype=torch.complex64)
    pad_f = torch.full((bsz, 1), x_f, dtype=torch.complex64)
    x_pad = torch.cat([pad_i, x, pad_f], dim=1)
    dx = x_pad[:, 1:] - x_pad[:, :-1]
    kinetic = (m_val / (2 * a_dt)) * torch.sum(dx**2, dim=1)
    omega_sq = omega_val**2 - 1j * eps_val
    pot_harm = (a_dt * m_val / 2) * omega_sq * torch.sum(x**2, dim=1)
    pot_quartic = a_dt * (lam_val / 4) * torch.sum(x**4, dim=1)
    return kinetic - pot_harm - pot_quartic

## 3. Nonlinear Thimble Model

The network maps real Gaussian noise $\mathbf{z} \in \mathbb{R}^D$ into **complex paths** $\mathbf{x} \in \mathbb{C}^D$ that live near the **thimble** — the complexified contour on which the imaginary part of the action is stationary and the arrows $e^{iS}$ align:

$$\mathbf{x} \;=\; A^\top \mathbf{z} \;+\; \mathbf{b} \;+\; g_\theta(\mathbf{z})$$

- $A$ is a complex $D\!\times\!D$ matrix (learnable real/imag parts) — the **linear thimble map**. Starts near the identity so the untrained network outputs real paths.
- $\mathbf{b}$ is a complex bias. `b_init` is set to the interior points of a linear interpolation between $x_i$ and $x_f$, so for **non-zero endpoints** the flow starts centred on the correct classical trajectory $x_{\text{cl}}(t)$.
- $g_\theta(\mathbf{z})$ is a **rank-$r$ bottleneck MLP** (two $\tanh$ layers with inner width $r \!\ll\! D$) that adds a nonlinear correction on top of the linear map. Its Jacobian is computed via the **matrix-determinant lemma**, making $\log\det J$ an $O(r^2)$ operation rather than the naive $O(D^3)$.

**Why a nonlinear correction?** For the harmonic oscillator a single linear $A$ is sufficient (the thimble is a plane). The quartic term $\tfrac{\lambda}{4}x^4$ curves the thimble, so a small nonlinear piece $g_\theta$ lets the network bend the contour where needed.

In [ ]:
class ThimbleFlowNL(nn.Module):
    def __init__(self, dim, rank=8, b_init=None):
        super().__init__()
        self.dim = dim
        self.rank = rank

        self.A_real = nn.Parameter(torch.eye(dim) + 0.01 * torch.randn(dim, dim))
        self.A_imag = nn.Parameter(0.01 * torch.randn(dim, dim))

        # bias init: linear interpolation for general endpoints, zeros for (0,0)
        if b_init is not None:
            self.b_real = nn.Parameter(b_init.clone().float())
        else:
            self.b_real = nn.Parameter(torch.zeros(dim))
        self.b_imag = nn.Parameter(torch.zeros(dim))

        r = rank
        self.W1 = nn.Parameter(torch.randn(r, dim) * (2.0 / dim)**0.5)
        self.b1 = nn.Parameter(torch.zeros(r))
        self.W2 = nn.Parameter(torch.randn(r, r) * (2.0 / r)**0.5)
        self.b2 = nn.Parameter(torch.zeros(r))
        self.W3 = nn.Parameter(torch.randn(2 * dim, r) * 0.001)
        self.b3 = nn.Parameter(torch.zeros(2 * dim))

    def _get_A(self):
        return self.A_real + 1j * self.A_imag

    def forward(self, z):
        A = self._get_A()
        b = self.b_real + 1j * self.b_imag
        x = z.to(torch.complex64) @ A.t() + b
        h1 = z @ self.W1.t() + self.b1
        a1 = torch.tanh(h1)
        h2 = a1 @ self.W2.t() + self.b2
        a2 = torch.tanh(h2)
        out = a2 @ self.W3.t() + self.b3
        g = out[:, :self.dim].to(torch.complex64) + 1j * out[:, self.dim:].to(torch.complex64)
        return x + g, a1, a2

    def compute_log_det_J(self, z, a1, a2, detach_correction=False):
        A = self._get_A()
        sign_A, logabs_A = torch.linalg.slogdet(A)
        log_det_A = logabs_A + 1j * torch.angle(sign_A)

        C = self.W3[:self.dim].to(torch.complex64) + 1j * self.W3[self.dim:].to(torch.complex64)
        A_invT_C = torch.linalg.solve(A.t(), C)
        K_mat = self.W1.to(torch.complex64) @ A_invT_C

        batch = z.shape[0]
        r = self.rank
        _a1 = a1.detach() if detach_correction else a1
        _a2 = a2.detach() if detach_correction else a2

        da1 = (1.0 - _a1**2).to(torch.complex64)
        M1 = da1.unsqueeze(-1) * K_mat.unsqueeze(0)
        W2c = self.W2.to(torch.complex64)
        if detach_correction:
            W2c = W2c.detach()
        M2 = torch.bmm(W2c.unsqueeze(0).expand(batch, -1, -1), M1)
        da2 = (1.0 - _a2**2).to(torch.complex64)
        M3 = da2.unsqueeze(-1) * M2
        I_r = torch.eye(r, dtype=torch.complex64).unsqueeze(0)
        sign_c, logabs_c = torch.linalg.slogdet(I_r + M3)
        log_det_correction = logabs_c + 1j * torch.angle(sign_c)
        if detach_correction:
            log_det_correction = log_det_correction.detach()
        return log_det_A + log_det_correction

## 4. Training & Estimation Function

### The importance weight

Changing variables from the real axis to the learned contour means every sample $\mathbf{z}$ carries a **complex log-weight**

$$\log W \;=\; i\,S\!\big[\mathbf{x}(\mathbf{z})\big] \;+\; \log\det J(\mathbf{z}) \;-\; \log P(\mathbf{z})$$

where $J$ is the Jacobian of the flow $\mathbf{z} \!\to\! \mathbf{x}$ and $P(\mathbf{z}) = \mathcal{N}(0, I)$ is the Gaussian base density. Each $W$ is an **arrow**: a complex number with magnitude and direction. If the arrows all point the same way, the path-integral sum is easy; if they scatter, we have the sign problem.

### The Monte-Carlo propagator

$$K(x_f, T;\, x_i, 0) \;=\; \Big(\tfrac{m}{2\pi i a}\Big)^{N/2}\; \mathbb{E}_{\mathbf{z}\sim P}\!\left[e^{\log W}\right] \;\approx\; \Big(\tfrac{m}{2\pi i a}\Big)^{N/2}\cdot\frac{1}{N_{\text{mc}}}\sum_{k=1}^{N_{\text{mc}}} e^{\log W_k}$$

Training minimises the **variance of $\log W$** so that the arrows align:

$$\mathcal{L} \;=\; \mathrm{Var}\!\big(\mathrm{Re}\,\log W\big) \;+\; \mathrm{Var}\!\big(\mathrm{Im}\,\log W\big)$$

The network parameters ($A$, $\mathbf{b}$, and the MLP weights in $g_\theta$) are updated by Adam; $\mathbf{z}$ is **resampled fresh** every epoch — it is not a learnable variable.

### Effective sample size

$$\mathrm{ESS}/N \;=\; \frac{\lvert\langle W \rangle\rvert^{2}}{\langle \lvert W \rvert^{2}\rangle}$$

$\mathrm{ESS}/N \!\to\! 1$ means arrows are fully aligned; a value near 0 means a few samples dominate (sign problem survives).

`run_single()` trains a fresh model for one $(\lambda, \text{seed})$ and returns:
- propagator $K$, MC std error, loss history, `ESS/N`
- the **trained model** (so we can re-evaluate with fresh samples later)
- a **sample of 20k complex `log W` values** (for bootstrap and weight-distribution diagnostics)
- 200 before/after arrow samples for plotting

In [ ]:
def run_single(lam_val, x_i=0.0, x_f=0.0,
               N_val=16, T_val=1.0, m_val=1.0, omega_val=2.0, eps_val=0.01,
               rank_val=8, epochs_val=15000, batch_val=2048, N_mc_val=100_000,
               seed=42, verbose=True):

    torch.manual_seed(seed)
    np.random.seed(seed)

    D_val = N_val - 1
    a_dt = T_val / N_val

    # bias init for general endpoints (zeros when x_i == x_f == 0)
    b_init = torch.linspace(float(x_i), float(x_f), D_val + 2)[1:-1]
    model = ThimbleFlowNL(D_val, rank=rank_val, b_init=b_init)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs_val, eta_min=1e-4)

    loss_history = []
    t0 = time.time()

    for epoch in range(epochs_val):
        optimizer.zero_grad()
        z = torch.randn(batch_val, D_val)
        x, a1, a2 = model(z)
        S = complex_action(x, x_i, x_f, lam_val, a_dt, m_val, omega_val, eps_val)
        log_P = -0.5 * torch.sum(z**2, dim=1) - (D_val / 2) * np.log(2 * np.pi)
        log_det_J = model.compute_log_det_J(z, a1, a2, detach_correction=True)
        log_W = 1j * S + log_det_J - log_P.to(torch.complex64)
        loss = torch.var(log_W.real) + torch.var(log_W.imag)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        scheduler.step()
        loss_history.append(loss.item())
        if verbose and ((epoch + 1) % 3000 == 0 or epoch == 0):
            print(f"    Epoch {epoch+1:5d}/{epochs_val} | loss = {loss.item():.6e} | "
                  f"{time.time()-t0:.0f}s", flush=True)

    elapsed = time.time() - t0

    norm_factor = (m_val / (2 * np.pi * 1j * a_dt)) ** (N_val / 2)
    with torch.no_grad():
        z_mc = torch.randn(N_mc_val, D_val)
        x_mc, a1_mc, a2_mc = model(z_mc)
        S_mc = complex_action(x_mc, x_i, x_f, lam_val, a_dt, m_val, omega_val, eps_val)
        log_P_mc = -0.5 * torch.sum(z_mc**2, dim=1) - (D_val / 2) * np.log(2 * np.pi)
        log_det_mc = model.compute_log_det_J(z_mc, a1_mc, a2_mc)
        log_W_mc = 1j * S_mc + log_det_mc - log_P_mc.to(torch.complex64)
        log_W_max = torch.max(log_W_mc.real)
        W_shifted = torch.exp(log_W_mc - log_W_max)
        K_mc = norm_factor * torch.exp(log_W_max).item() * torch.mean(W_shifted).numpy()
        W_for_err = norm_factor * torch.exp(log_W_max).item() * W_shifted.numpy()
        K_std = np.std(W_for_err) / np.sqrt(N_mc_val)
        W_ess = torch.exp(log_W_mc[:10000] - torch.max(log_W_mc[:10000].real)).numpy()
        ess = float(np.abs(np.mean(W_ess))**2 / np.mean(np.abs(W_ess)**2))

        # keep a 20k subsample of the raw complex log-weights for later
        # weight-distribution diagnostics (log|W|, arg W) and bootstrap CIs.
        log_W_sample = log_W_mc[:20000].detach().cpu().numpy().astype(np.complex128)

    # ── arrow data (200 samples) ──
    n_arrows = 200
    with torch.no_grad():
        z_arr = torch.randn(n_arrows, D_val)
        S_before = complex_action(z_arr.to(torch.complex64),
                                  x_i, x_f, lam_val, a_dt, m_val, omega_val, eps_val)
        W_before = torch.exp(1j * S_before).numpy()
        W_before_norm = W_before / np.max(np.abs(W_before))

        x_arr, a1_arr, a2_arr = model(z_arr)
        S_after = complex_action(x_arr, x_i, x_f, lam_val, a_dt, m_val, omega_val, eps_val)
        log_P_arr = -0.5 * torch.sum(z_arr**2, dim=1) - (D_val / 2) * np.log(2 * np.pi)
        log_det_arr = model.compute_log_det_J(z_arr, a1_arr, a2_arr)
        log_W_arr = 1j * S_after + log_det_arr - log_P_arr.to(torch.complex64)
        W_after = torch.exp(log_W_arr - torch.max(log_W_arr.real)).numpy()
        W_after_norm = W_after / np.max(np.abs(W_after))

    result = dict(
        lam=lam_val, x_i=x_i, x_f=x_f, seed=seed,
        N=N_val, T=T_val, m=m_val,
        omega=omega_val, eps=eps_val, rank=rank_val, epochs=epochs_val,
        K_re=float(K_mc.real), K_im=float(K_mc.imag), abs_K=float(abs(K_mc)),
        std_err=float(K_std), final_loss=loss_history[-1], ess_n=ess,
        time_s=elapsed, loss_history=loss_history,
        W_before_norm=W_before_norm, W_after_norm=W_after_norm,
        # extras used by the validation / diagnostic cells below
        model=model,
        log_W_sample=log_W_sample,
        norm_factor_complex=complex(norm_factor),
        a_dt=float(a_dt), D=int(D_val),
    )
    if verbose:
        print(f"  => K = {K_mc:.10f}  |K| = {abs(K_mc):.8f}  "
              f"loss = {loss_history[-1]:.2e}  ESS/N = {ess:.4f}  ({elapsed:.0f}s)",
              flush=True)
    return result

## 5. Exact & Independent Solutions

- **Mehler kernel** — exact HO propagator for any endpoints
- **Free-particle kernel** — exact with $\omega=0$, $\lambda=0$
- **Basis truncation** — independent numerical verification

In [ ]:
def mehler_kernel(x_i, x_f, T_val, m_val, omega_val):
    sinwT = np.sin(omega_val * T_val)
    coswT = np.cos(omega_val * T_val)
    prefactor = np.sqrt(m_val * omega_val / (2 * np.pi * 1j * sinwT))
    exponent = (1j * m_val * omega_val / (2 * sinwT)) * (
        (x_i**2 + x_f**2) * coswT - 2 * x_i * x_f)
    return prefactor * np.exp(exponent)

def free_particle_kernel(x_i, x_f, T_val, m_val):
    return np.sqrt(m_val / (2 * np.pi * 1j * T_val)) * np.exp(
        1j * m_val * (x_f - x_i)**2 / (2 * T_val))

# ── Basis truncation (independent numerical verification) ──
N_basis = 400
alpha_x = 1.0 / np.sqrt(2 * m * omega)

def _psi_n_HO(n, x):
    xi = np.sqrt(m * omega) * x
    gauss_pref = (m * omega / np.pi)**0.25 * np.exp(-0.5 * xi**2)
    if n == 0:
        return gauss_pref
    psi_prev = 1.0
    psi_curr = np.sqrt(2) * xi
    if n == 1:
        return gauss_pref * psi_curr
    for k in range(1, n):
        psi_next = (np.sqrt(2)*xi*psi_curr - np.sqrt(k)*psi_prev) / np.sqrt(k+1)
        psi_prev = psi_curr
        psi_curr = psi_next
    return gauss_pref * psi_curr

def _x4_HO_matrix(Nb):
    x2 = np.zeros((Nb, Nb))
    for nn in range(Nb):
        x2[nn, nn] = alpha_x**2 * (2*nn + 1)
        if nn + 2 < Nb:
            val = alpha_x**2 * np.sqrt((nn+1)*(nn+2))
            x2[nn, nn+2] = val
            x2[nn+2, nn] = val
    return x2 @ x2

def compute_K_basis_truncation(xi, xf, lam_val, T_val, Nb=400):
    E_HO = np.array([omega * (n + 0.5) for n in range(Nb)])
    phi_xi = np.array([_psi_n_HO(n, xi) for n in range(Nb)])
    phi_xf = np.array([_psi_n_HO(n, xf) for n in range(Nb)])
    x4_mat = _x4_HO_matrix(Nb)
    H = np.diag(E_HO) + (lam_val / 4.0) * x4_mat
    E_vals, U = np.linalg.eigh(H)
    psi_xi_eig = U.T @ phi_xi
    psi_xf_eig = U.T @ phi_xf
    weights = psi_xf_eig * psi_xi_eig
    etas = np.array([0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.1, 0.12, 0.15])
    K_eta = np.array([np.sum(weights * np.exp(-E_vals * (eta + 1j*T_val)))
                      for eta in etas])
    cr4 = np.polyfit(etas, K_eta.real, 4)
    ci4 = np.polyfit(etas, K_eta.imag, 4)
    K4 = np.polyval(cr4, 0) + 1j * np.polyval(ci4, 0)
    cr3 = np.polyfit(etas, K_eta.real, 3)
    ci3 = np.polyfit(etas, K_eta.imag, 3)
    K3 = np.polyval(cr3, 0) + 1j * np.polyval(ci3, 0)
    return K4, abs(K4 - K3)

K_mehler_ref = mehler_kernel(x_i_prop, x_f_prop, T, m, omega)
K_free_ref  = free_particle_kernel(x_i_prop, x_f_prop, T, m)
print(f"Mehler K({x_f_prop},T;{x_i_prop},0) = {K_mehler_ref:.10f}")
print(f"Free particle K({x_f_prop},T;{x_i_prop},0) = {K_free_ref:.10f}")

## 6. Lambda Scan (multi-seed)

For each $\lambda$ in `lambda_list` we train a **fresh** neural network for **every seed** in `seed_list` (so we can measure training-noise spread later). Each trained model is fed 100k fresh Monte-Carlo samples to estimate $K$. The seed with the **lowest final variance loss** is marked as `is_best=True` and promoted into `best_results`; every run (including the non-best ones) is kept in `all_results` for the multi-seed stability study in Section&nbsp;9.

Why multiple seeds? Training is stochastic (random $\mathbf{z}$ each epoch, random weight init). If different seeds produce wildly different $K$, the answer depends on training — not on physics. If they cluster tightly (within each run's MC error bar), the ML estimator is self-consistent.

In [ ]:
#seed_list = [42, 45, 99, 123, 2024]   # 5 random seeds per lambda for stability analysis
seed_list = [42] 
all_results = []      # every run (len(seed_list) per lambda)
best_results = []     # one per lambda (lowest final_loss)

print("=" * 80)
print(f"LAMBDA SCAN — NN Thimble + Basis Truncation  (x_i={x_i_prop}, x_f={x_f_prop})")
print(f"  Lambdas: {lambda_list}")
print(f"  Seeds per lambda: {seed_list}")
print("=" * 80)

for i, lam_val in enumerate(lambda_list):
    print(f"\n{'='*80}")
    print(f"[{i+1}/{len(lambda_list)}] lam = {lam_val}")
    print(f"{'='*80}", flush=True)

    # ── Basis truncation (independent, computed once per lambda) ──
    K_basis, K_basis_unc = compute_K_basis_truncation(
        x_i_prop, x_f_prop, lam_val, T, N_basis)
    K_meh = mehler_kernel(x_i_prop, x_f_prop, T, m, omega)

    seed_runs = []
    for s_idx, seed_val in enumerate(seed_list):
        print(f"\n  --- Seed {seed_val} ({s_idx+1}/{len(seed_list)}) ---", flush=True)

        res = run_single(lam_val, x_i=x_i_prop, x_f=x_f_prop,
                         N_val=N, T_val=T, m_val=m, omega_val=omega, eps_val=epsilon,
                         rank_val=rank, epochs_val=epochs, batch_val=batch,
                         N_mc_val=N_mc, seed=seed_val)

        res['K_basis_re'] = float(K_basis.real)
        res['K_basis_im'] = float(K_basis.imag)
        res['abs_K_basis'] = float(abs(K_basis))
        res['K_basis_unc'] = float(K_basis_unc)
        res['K_mehler_re'] = float(K_meh.real)
        res['K_mehler_im'] = float(K_meh.imag)

        K_ml = res['K_re'] + 1j * res['K_im']
        rel_diff = abs(K_ml - K_basis) / max(abs(K_basis), 1e-30)
        res['rel_diff_ml_basis'] = float(rel_diff)

        seed_runs.append(res)
        all_results.append(res)

    # ── Pick best seed (lowest final_loss) ──
    best = min(seed_runs, key=lambda r: r['final_loss'])
    best['is_best'] = True
    for r in seed_runs:
        if r is not best:
            r['is_best'] = False
    best_results.append(best)

    # ── Summary table for this lambda ──
    print(f"\n  Basis truncation: {K_basis:.10f}  (unc={K_basis_unc:.2e})")
    print(f"\n  {'Seed':>6s}  {'K_ML':>30s}  {'|K|':>10s}  {'loss':>10s}  {'ESS':>6s}  {'rel_diff':>9s}  Best?")
    print(f"  {'-'*90}")
    for r in seed_runs:
        kml = f"{r['K_re']:+.8f}{r['K_im']:+.8f}j"
        tag = " <<<" if r.get('is_best') else ""
        print(f"  {r['seed']:6d}  {kml:>30s}  {r['abs_K']:10.6f}  "
              f"{r['final_loss']:10.2e}  {r['ess_n']:6.4f}  {r['rel_diff_ml_basis']:9.2e}{tag}")

# keep scan_results pointing to best for downstream cells
scan_results = best_results

### Save scan results to CSV

In [ ]:
csv_path = f"nn_lambda_scan_xi{x_i_prop}_xf{x_f_prop}.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["lam","x_i","x_f","seed","best",
                "K_ml_re","K_ml_im","abs_K_ml","std_err",
                "final_loss","ess_n",
                "K_basis_re","K_basis_im","abs_K_basis","K_basis_unc",
                "rel_diff_ml_basis",
                "K_mehler_re","K_mehler_im",
                "N","T","m","omega","eps","rank","epochs","time_s"])
    for r in all_results:
        w.writerow([
            f"{r['lam']:.6g}", f"{r['x_i']:.4g}", f"{r['x_f']:.4g}",
            r['seed'], "Y" if r.get('is_best') else "N",
            f"{r['K_re']:.16e}", f"{r['K_im']:.16e}", f"{r['abs_K']:.16e}",
            f"{r['std_err']:.6e}", f"{r['final_loss']:.6e}", f"{r['ess_n']:.6f}",
            f"{r['K_basis_re']:.16e}", f"{r['K_basis_im']:.16e}",
            f"{r['abs_K_basis']:.16e}", f"{r['K_basis_unc']:.6e}",
            f"{r['rel_diff_ml_basis']:.6e}",
            f"{r['K_mehler_re']:.16e}", f"{r['K_mehler_im']:.16e}",
            r['N'], r['T'], r['m'], r['omega'], r['eps'], r['rank'],
            r['epochs'], f"{r['time_s']:.1f}"])

print(f"Saved {len(all_results)} rows ({len(best_results)} lambdas × {len(seed_list)} seeds) to {csv_path}")
print()
hdr = f"{'lam':>5s}  {'seed':>6s}  {'best':>4s}  {'K_ML':>30s}  {'K_basis':>30s}  {'rel_diff':>9s}  {'loss':>9s}  {'ESS':>6s}"
print(hdr)
print("-" * len(hdr))
for r in all_results:
    kml = f"{r['K_re']:+.8f}{r['K_im']:+.8f}j"
    kba = f"{r['K_basis_re']:+.8f}{r['K_basis_im']:+.8f}j"
    tag = " <<<" if r.get('is_best') else ""
    print(f"{r['lam']:5.2f}  {r['seed']:6d}  {'Y' if r.get('is_best') else 'N':>4s}  "
          f"{kml:>30s}  {kba:>30s}  {r['rel_diff_ml_basis']:9.2e}  "
          f"{r['final_loss']:9.2e}  {r['ess_n']:6.4f}{tag}")

### Loss curves (6 per page)

In [ ]:
n_per_page = 6
n_pages = math.ceil(len(scan_results) / n_per_page)
for pg in range(n_pages):
    subset = scan_results[pg*n_per_page : (pg+1)*n_per_page]
    nrows = math.ceil(len(subset) / 3)
    ncols = min(len(subset), 3)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), squeeze=False)
    for idx, r in enumerate(subset):
        row, col = divmod(idx, 3)
        ax = axes[row][col]
        ax.semilogy(r['loss_history'])
        ax.set_title(f"lam={r['lam']:.2f}  loss={r['final_loss']:.2e}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.3)
    for idx in range(len(subset), nrows*ncols):
        row, col = divmod(idx, 3)
        axes[row][col].set_visible(False)
    plt.tight_layout()
    fig.savefig(f"loss_curves_page{pg+1}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved loss_curves_page{pg+1}.png")

### Arrow diagrams — AFTER training (6 per page)

Shows learned-thimble importance-weight arrows for each $\lambda$.

In [ ]:
for pg in range(n_pages):
    subset = scan_results[pg*n_per_page : (pg+1)*n_per_page]
    nrows = math.ceil(len(subset) / 3)
    ncols = min(len(subset), 3)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows), squeeze=False)
    for idx, r in enumerate(subset):
        row, col = divmod(idx, 3)
        ax = axes[row][col]
        W = r['W_after_norm']
        for i in range(len(W)):
            ax.annotate('', xy=(W[i].real, W[i].imag), xytext=(0,0),
                        arrowprops=dict(arrowstyle='->', color='crimson', alpha=0.2, lw=0.9))
        ax.add_patch(plt.Circle((0,0), 1, fill=False, color='gray', ls='--', alpha=0.3))
        ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
        ax.set_aspect('equal')
        ax.set_title(f"AFTER  lam={r['lam']:.2f}  ESS={r['ess_n']:.3f}")
        ax.grid(True, alpha=0.2)
    for idx in range(len(subset), nrows*ncols):
        row, col = divmod(idx, 3)
        axes[row][col].set_visible(False)
    plt.tight_layout()
    fig.savefig(f"arrows_after_page{pg+1}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved arrows_after_page{pg+1}.png")

## 7. Sanity Check: $\lambda=0$ — ML vs Basis Truncation vs Mehler

All three methods should agree at $\lambda=0$ for the chosen endpoints $(x_i, x_f)$.

In [ ]:
print("=" * 70)
print(f"SANITY CHECK: lam=0, ({x_i_prop},{x_f_prop}) — ML vs Basis vs Mehler")
print("=" * 70)

r0 = [r for r in scan_results if abs(r['lam']) < 1e-12]
if r0:
    r0 = r0[0]
    K_ml  = r0['K_re'] + 1j * r0['K_im']
    K_bas = r0['K_basis_re'] + 1j * r0['K_basis_im']
    K_meh = mehler_kernel(x_i_prop, x_f_prop, T, m, omega)

    err_ml_meh  = abs(K_ml  - K_meh) / abs(K_meh)
    err_bas_meh = abs(K_bas - K_meh) / abs(K_meh)
    err_ml_bas  = abs(K_ml  - K_bas) / abs(K_bas)

    print(f"  Mehler (exact):     {K_meh:.10f}")
    print(f"  ML NN thimble:     {K_ml:.10f}   rel err vs Mehler: {err_ml_meh:.4e}")
    print(f"  Basis truncation:  {K_bas:.10f}   rel err vs Mehler: {err_bas_meh:.4e}")
    print(f"  ML vs Basis:       rel diff = {err_ml_bas:.4e}")
    print()
    all_pass = err_ml_meh < 0.02 and err_bas_meh < 0.01 and err_ml_bas < 0.02
    print(f"  >> {'ALL THREE AGREE — PASSED' if all_pass else 'DISCREPANCY — CHECK'}")
else:
    print("  lam=0 not in scan! Add 0.0 to lambda_list.")

### Sanity check: $\lambda = 0.05$ — ML vs Basis should be very close

In [ ]:
r005 = [r for r in scan_results if abs(r['lam'] - 0.05) < 1e-6]
if r005:
    r005 = r005[0]
    K_ml  = r005['K_re'] + 1j * r005['K_im']
    K_bas = r005['K_basis_re'] + 1j * r005['K_basis_im']
    rel = abs(K_ml - K_bas) / abs(K_bas)
    print(f"lam=0.05 sanity:")
    print(f"  ML:    {K_ml:.10f}")
    print(f"  Basis: {K_bas:.10f}")
    print(f"  rel diff: {rel:.4e}  >> {'PASSED' if rel < 0.05 else 'CHECK'}")
else:
    print("  lam=0.05 not in scan!")

## 8. Physics Sanity Checks

Three independent physics checks that do **not** require large endpoints:

1. **Mehler agreement at $\lambda=0$** — ML and Basis must match the exact
   Mehler kernel for the chosen $(x_i, x_f)$.
2. **Short-time limit** — for small $T$, the propagator approaches the
   free-particle kernel regardless of $\lambda$ or $\omega$.
3. **Endpoint swap symmetry** — $K(x_f, T;\, x_i, 0) = K(x_i, T;\, x_f, 0)$
   for the time-reversal-invariant Hamiltonian.

In [ ]:
run_physics_checks = True

# ── CHECK 1: Mehler agreement at λ=0 ──
print("=" * 70)
print(f"CHECK 1 — Mehler agreement at λ=0  ({x_i_prop}, {x_f_prop})")
print("=" * 70)

r0 = [r for r in best_results if abs(r['lam']) < 1e-12]
if r0:
    r0 = r0[0]
    K_ml  = r0['K_re'] + 1j * r0['K_im']
    K_bas = r0['K_basis_re'] + 1j * r0['K_basis_im']
    K_meh = mehler_kernel(x_i_prop, x_f_prop, T, m, omega)

    err_ml  = abs(K_ml  - K_meh) / abs(K_meh)
    err_bas = abs(K_bas - K_meh) / abs(K_meh)

    print(f"  Mehler (exact):    {K_meh:.10f}")
    print(f"  ML (best seed):   {K_ml:.10f}   rel err = {err_ml:.4e}")
    print(f"  Basis truncation: {K_bas:.10f}   rel err = {err_bas:.4e}")
    passed_1 = err_ml < 0.02 and err_bas < 0.01
    print(f"  >> {'PASSED' if passed_1 else 'FAILED'}")
else:
    print("  λ=0 not in scan — add 0.0 to lambda_list to enable this check.")
    passed_1 = None

# ── CHECK 2: Short-time limit ──
print()
print("=" * 70)
print(f"CHECK 2 — Short-time limit  ({x_i_prop}, {x_f_prop}), T_short=0.01")
print("=" * 70)

if run_physics_checks:
    T_short = 0.01
    lam_test = 2.0
    res_short = run_single(lam_test,
                           x_i=x_i_prop, x_f=x_f_prop,
                           N_val=N, T_val=T_short, m_val=m,
                           omega_val=omega, eps_val=epsilon,
                           rank_val=rank, epochs_val=epochs,
                           batch_val=batch, N_mc_val=N_mc,
                           seed=42, verbose=False)
    K_ml_short   = res_short['K_re'] + 1j * res_short['K_im']
    K_free_short = free_particle_kernel(x_i_prop, x_f_prop, T_short, m)
    err_short = abs(K_ml_short - K_free_short) / abs(K_free_short)
    print(f"  Free-particle (exact): {K_free_short:.10f}")
    print(f"  ML (λ={lam_test}, T={T_short}): {K_ml_short:.10f}")
    print(f"  Relative error: {err_short:.4e}")
    passed_2 = err_short < 0.05
    print(f"  >> {'PASSED' if passed_2 else 'FAILED'}")
else:
    passed_2 = None

# ── CHECK 3: Endpoint swap symmetry ──
print()
print("=" * 70)
print(f"CHECK 3 — Endpoint swap symmetry  K({x_f_prop},T;{x_i_prop},0) vs K({x_i_prop},T;{x_f_prop},0)")
print("=" * 70)

if run_physics_checks and x_i_prop != x_f_prop:
    lam_swap = 1.0
    res_fwd = run_single(lam_swap,
                         x_i=x_i_prop, x_f=x_f_prop,
                         N_val=N, T_val=T, m_val=m,
                         omega_val=omega, eps_val=epsilon,
                         rank_val=rank, epochs_val=epochs,
                         batch_val=batch, N_mc_val=N_mc,
                         seed=42, verbose=False)
    res_rev = run_single(lam_swap,
                         x_i=x_f_prop, x_f=x_i_prop,
                         N_val=N, T_val=T, m_val=m,
                         omega_val=omega, eps_val=epsilon,
                         rank_val=rank, epochs_val=epochs,
                         batch_val=batch, N_mc_val=N_mc,
                         seed=42, verbose=False)
    K_fwd = res_fwd['K_re'] + 1j * res_fwd['K_im']
    K_rev = res_rev['K_re'] + 1j * res_rev['K_im']
    err_swap = abs(K_fwd - K_rev) / max(abs(K_fwd), abs(K_rev), 1e-30)
    print(f"  K({x_f_prop},T;{x_i_prop},0) = {K_fwd:.10f}")
    print(f"  K({x_i_prop},T;{x_f_prop},0) = {K_rev:.10f}")
    print(f"  Relative difference: {err_swap:.4e}")
    passed_3 = err_swap < 0.05
    print(f"  >> {'PASSED' if passed_3 else 'FAILED'}")
elif x_i_prop == x_f_prop:
    print("  Endpoints are equal — swap symmetry is trivially satisfied.")
    passed_3 = True
else:
    passed_3 = None

# ── Summary ──
print()
print("=" * 70)
print("PHYSICS CHECKS SUMMARY")
print("=" * 70)
labels = ["Mehler at λ=0", "Short-time limit", "Endpoint swap"]
results = [passed_1, passed_2, passed_3]
for lbl, res in zip(labels, results):
    status = "PASSED" if res is True else ("FAILED" if res is False else "SKIPPED")
    print(f"  {lbl:25s}  {status}")

## 9. Comparison Plot: ML Best Seed vs Basis Truncation

For each $\lambda$, plots the propagator components ($\mathrm{Re}\,K$, $\mathrm{Im}\,K$, $|K|$)
from the **best-seed ML run** alongside the **basis truncation** estimate.

In [ ]:
lams_best     = np.array([r['lam']        for r in best_results])
ml_re_best    = np.array([r['K_re']       for r in best_results])
ml_im_best    = np.array([r['K_im']       for r in best_results])
ml_abs_best   = np.array([r['abs_K']      for r in best_results])
basis_re      = np.array([r['K_basis_re'] for r in best_results])
basis_im      = np.array([r['K_basis_im'] for r in best_results])
basis_abs     = np.array([r['abs_K_basis'] for r in best_results])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Re(K) ──
ax = axes[0]
ax.plot(lams_best, ml_re_best,  'o-', color='crimson',   label='ML (best seed)')
ax.plot(lams_best, basis_re,    's--', color='steelblue', label='Basis truncation')
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel(r'$\mathrm{Re}\,K$')
ax.set_title(r'Real part of propagator')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Im(K) ──
ax = axes[1]
ax.plot(lams_best, ml_im_best,  'o-', color='crimson',   label='ML (best seed)')
ax.plot(lams_best, basis_im,    's--', color='steelblue', label='Basis truncation')
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel(r'$\mathrm{Im}\,K$')
ax.set_title(r'Imaginary part of propagator')
ax.legend()
ax.grid(True, alpha=0.3)

# ── |K| ──
ax = axes[2]
ax.plot(lams_best, ml_abs_best,  'o-', color='crimson',   label='ML (best seed)')
ax.plot(lams_best, basis_abs,    's--', color='steelblue', label='Basis truncation')
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel(r'$|K|$')
ax.set_title(r'Magnitude of propagator')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'ML Propagator (best seed) vs Basis Truncation — K({x_f_prop},T;{x_i_prop},0)',
             fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig("ml_vs_basis_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved ml_vs_basis_comparison.png")

## 10. Basis Truncation Convergence Check (independent of ML)

Sweep the basis size $N_b$ for several values of $\lambda$ and watch the
basis-truncation propagator $K$ plateau. This is purely a numerical check
on `compute_K_basis_truncation` — it does **not** use the trained neural
network.

For each $\lambda$ we compute $K(N_b)$ at a list of basis sizes, plot
$\mathrm{Re}\,K,\ \mathrm{Im}\,K,\ |K|$ vs $N_b$, and print a **CONVERGED /
NOT CONVERGED** verdict based on whether the last few $N_b$ values agree
with the largest-$N_b$ reference within a relative tolerance.

At $\lambda=0$ the exact Mehler kernel is overlaid as a sanity reference.

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt

conv_lambda_list = [0.0, 0.05, 0.5, 1.0, 2.0]

nb_list = [25, 50, 100, 150, 200, 300, 400, 600, 800]

rel_tol = 1e-5

conv_results       = []
overall_verdicts   = {}

print("=" * 92)
print("BASIS TRUNCATION CONVERGENCE CHECK  (independent of ML)")
print(f"  Endpoints: x_i={x_i_prop}, x_f={x_f_prop}   T={T}   omega={omega}   m={m}")
print(f"  Nb sweep:  {nb_list}")
print(f"  Lambdas:   {conv_lambda_list}")
print(f"  Tolerance: |K(Nb)-K(Nb_max)| / |K(Nb_max)| < {rel_tol:.1e}  on last 3 Nb")
print("=" * 92)

for lam_val in conv_lambda_list:
    print(f"\n---- lambda = {lam_val} ----")

    K_arr   = []
    unc_arr = []
    for Nb in nb_list:
        K_nb, K_unc = compute_K_basis_truncation(x_i_prop, x_f_prop, lam_val, T, Nb)
        K_arr.append(K_nb)
        unc_arr.append(K_unc)
    K_arr   = np.array(K_arr)
    unc_arr = np.array(unc_arr)
    K_ref   = K_arr[-1]

    K_meh = mehler_kernel(x_i_prop, x_f_prop, T, m, omega) if abs(lam_val) < 1e-12 else None

    header = (f"  {'Nb':>5}  {'Re K':>13}  {'Im K':>13}  {'|K|':>10}  "
              f"{'|dK|':>10}  {'rel_to_ref':>10}  {'fit_unc':>10}")
    print(header)
    print("  " + "-" * (len(header) - 2))

    prev_K          = None
    converged_flags = []
    for i, Nb in enumerate(nb_list):
        K   = K_arr[i]
        dK  = abs(K - prev_K) if prev_K is not None else float('nan')
        rel = abs(K - K_ref) / max(abs(K_ref), 1e-30)
        flag = rel < rel_tol
        converged_flags.append(flag)
        tag = "  OK" if flag else ""
        print(f"  {Nb:5d}  {K.real:+13.8f}  {K.imag:+13.8f}  {abs(K):10.6f}  "
              f"{dK:10.2e}  {rel:10.2e}  {unc_arr[i]:10.2e}{tag}")
        prev_K = K
        conv_results.append(dict(
            lam=lam_val, Nb=Nb,
            K_re=float(K.real), K_im=float(K.imag), abs_K=float(abs(K)),
            abs_dK=None if np.isnan(dK) else float(dK),
            rel_to_ref=float(rel), fit_unc=float(unc_arr[i]),
            converged=bool(flag),
        ))

    last_n = min(3, len(converged_flags))
    is_converged = all(converged_flags[-last_n:])
    overall_verdicts[lam_val] = is_converged
    verdict_str = "CONVERGED" if is_converged else "NOT CONVERGED"
    print(f"  -> Verdict for lam={lam_val}: {verdict_str}  "
          f"(last {last_n} sizes within < {rel_tol:.0e} of Nb={nb_list[-1]})")
    if K_meh is not None:
        err_meh = abs(K_ref - K_meh) / abs(K_meh)
        print(f"     Mehler ref (lam=0): K={K_meh:.10f}   "
              f"rel err at Nb={nb_list[-1]}: {err_meh:.2e}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    re_vals = K_arr.real
    im_vals = K_arr.imag
    ab_vals = np.abs(K_arr)

    for ax, vals, name in zip(axes, [re_vals, im_vals, ab_vals],
                              ['Re K', 'Im K', '|K|']):
        ax.plot(nb_list, vals, 'o-', color='steelblue', label='basis trunc.')
        ax.axhline(vals[-1], color='crimson', ls='--', alpha=0.6,
                   label=f'ref (Nb={nb_list[-1]})')
        if K_meh is not None:
            ref = K_meh.real if name == 'Re K' else (K_meh.imag if name == 'Im K' else abs(K_meh))
            ax.axhline(ref, color='black', ls=':', label='Mehler (exact)')
        ax.set_xscale('log')
        ax.set_xlabel(r'$N_b$ (basis size)')
        ax.set_ylabel(name)
        ax.set_title(f'{name} vs $N_b$')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    plt.suptitle(
        f'Basis truncation convergence — $\\lambda$ = {lam_val}    ({verdict_str})',
        fontsize=13, y=1.02)
    plt.tight_layout()
    fname = f"basis_truncation_convergence_lam{lam_val}.png"
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"     Saved {fname}")

csv_path = "basis_truncation_convergence.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["lam", "Nb", "K_re", "K_im", "abs_K",
                "abs_dK", "rel_to_ref", "fit_unc", "converged"])
    for r in conv_results:
        w.writerow([
            r['lam'], r['Nb'],
            f"{r['K_re']:.10e}", f"{r['K_im']:.10e}", f"{r['abs_K']:.10e}",
            "" if r['abs_dK'] is None else f"{r['abs_dK']:.4e}",
            f"{r['rel_to_ref']:.4e}", f"{r['fit_unc']:.4e}",
            int(r['converged']),
        ])
print(f"\nSaved {len(conv_results)} rows to {csv_path}")

In [ ]:
print("=" * 80)
print("BASIS TRUNCATION CONVERGENCE  —  FINAL SUMMARY")
print("=" * 80)

n_ok  = sum(1 for v in overall_verdicts.values() if v)
n_tot = len(overall_verdicts)
print(f"  {n_ok} / {n_tot} lambda values converged at Nb = {nb_list[-1]}")
print(f"  Criterion: |K(Nb) - K(Nb_max)| / |K(Nb_max)| < {rel_tol:.0e} "
      f"for the last 3 Nb points")
print()
for lam_val in conv_lambda_list:
    status = "CONVERGED" if overall_verdicts[lam_val] else "NOT CONVERGED"
    print(f"    lam = {lam_val:<6}  ->  {status}")
print()

if n_ok == n_tot:
    print(f">>> OVERALL: CONVERGED at Nb = {nb_list[-1]} for all tested lambdas.")
    if N_basis >= nb_list[-1]:
        print(f"    Current notebook value N_basis = {N_basis} is sufficient.")
    else:
        print(f"    Current notebook value N_basis = {N_basis}; "
              f"it already exceeds the smallest fully-converged size, so it is fine.")
else:
    bad = [l for l, v in overall_verdicts.items() if not v]
    print(f">>> OVERALL: NOT CONVERGED for lambdas {bad} at Nb = {nb_list[-1]}.")
    print(f"    Recommend: extend nb_list beyond {nb_list[-1]} and re-run "
          f"(quartic term mixes high-n basis states more strongly at large lambda).")

## 11. ML Validation (independent of basis truncation)

The following cells test the **ML propagator on its own terms** — no cross-checks against basis truncation, no exact-kernel references. The logic is:

- If $K$ is really converged, it must be **stable** under the things that should not change the answer: random training seed, number of Monte-Carlo samples, and which particular 100k batch we average.
- The **weight distribution** itself must look healthy — not just have a favourable ESS number. A narrow unimodal $\log|W|$ histogram and a tight unimodal $\arg W$ histogram are strong evidence that the sampler is actually working and no single path is dominating.

Three checks are bundled here (per $\lambda$):

1. **Multi-seed stability** — 5 random seeds per $\lambda$ trained independently; spread of $K$ across seeds vs each run's own MC error bar.
2. **MC self-consistency** — for the best model per $\lambda$: $K$ vs $N_{\text{mc}}$ scaling, independent-batch spread, and a bootstrap confidence interval.
3. **Honest weight diagnostics** — ESS, $\mathrm{Var}(\log|W|)$, $\max|W|/\mathrm{mean}|W|$, plus histograms of $\log|W|$ and $\arg W$.

### 11.1 Multi-seed stability

For each $\lambda$ we trained $|\text{seed\_list}|$ independent models. Below we plot, per $\lambda$:

- every seed's $K$ (Re, Im, $|K|$) with its own MC error bar,
- the seed with the lowest final loss highlighted as a gold star ("best"),
- a shaded band showing the mean ± std **across seeds**.

**Interpretation.** If all seeds overlap within their own MC error bars, training noise is sub-dominant and the quoted best-seed $K$ is representative of what the ML estimator produces. If the across-seed spread is *much larger* than each run's MC error bar, $K$ depends on training luck — in that case you should retrain longer or average across seeds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lams_for_spread = sorted({r['lam'] for r in all_results})

print("=" * 100)
print("11.1  MULTI-SEED STABILITY  (5 seeds per lambda)")
print("=" * 100)

spread_summary = []

for lam_val in lams_for_spread:
    rs = [r for r in all_results if abs(r['lam'] - lam_val) < 1e-12]
    rs = sorted(rs, key=lambda r: r['seed'])

    seeds      = np.array([r['seed']        for r in rs])
    K_re_arr   = np.array([r['K_re']        for r in rs])
    K_im_arr   = np.array([r['K_im']        for r in rs])
    K_std_arr  = np.array([r['std_err']     for r in rs])
    loss_arr   = np.array([r['final_loss']  for r in rs])
    ess_arr    = np.array([r['ess_n']       for r in rs])
    absK_arr   = np.sqrt(K_re_arr**2 + K_im_arr**2)

    best_idx = int(np.argmin(loss_arr))
    best_seed = seeds[best_idx]

    re_mean, re_std = K_re_arr.mean(), K_re_arr.std(ddof=1)
    im_mean, im_std = K_im_arr.mean(), K_im_arr.std(ddof=1)
    ab_mean, ab_std = absK_arr.mean(), absK_arr.std(ddof=1)

    mean_mc_err = K_std_arr.mean()
    ratio_re = re_std / max(mean_mc_err, 1e-30)
    ratio_im = im_std / max(mean_mc_err, 1e-30)

    print(f"\n--- lambda = {lam_val} ---")
    print(f"  {'seed':>6}  {'Re K':>14}  {'Im K':>14}  {'|K|':>10}  "
          f"{'MC std':>10}  {'loss':>10}  {'ESS/N':>8}  best?")
    for i in range(len(rs)):
        mark = "  <-- best" if i == best_idx else ""
        print(f"  {seeds[i]:>6d}  {K_re_arr[i]:+14.8f}  {K_im_arr[i]:+14.8f}  "
              f"{absK_arr[i]:10.6f}  {K_std_arr[i]:10.2e}  "
              f"{loss_arr[i]:10.2e}  {ess_arr[i]:8.4f}{mark}")
    print(f"  across-seed  mean Re = {re_mean:+.6f}  std Re = {re_std:.2e}  "
          f"(std/MC_err = {ratio_re:.2f})")
    print(f"  across-seed  mean Im = {im_mean:+.6f}  std Im = {im_std:.2e}  "
          f"(std/MC_err = {ratio_im:.2f})")
    if max(ratio_re, ratio_im) < 3:
        verdict = "OK: seed spread is comparable to MC error -> stable"
    elif max(ratio_re, ratio_im) < 10:
        verdict = "MARGINAL: seed spread a few x the MC error -> consider more epochs"
    else:
        verdict = "UNSTABLE: seed spread >> MC error -> training noise dominates"
    print(f"  verdict: {verdict}")

    spread_summary.append(dict(
        lam=lam_val, seeds=seeds, best_seed=int(best_seed),
        K_re=K_re_arr, K_im=K_im_arr, abs_K=absK_arr, K_std=K_std_arr,
        re_mean=re_mean, re_std=re_std,
        im_mean=im_mean, im_std=im_std,
        ab_mean=ab_mean, ab_std=ab_std,
        ratio_re=ratio_re, ratio_im=ratio_im,
    ))

lams_plot = np.array([s['lam'] for s in spread_summary])
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
labels = ['Re K', 'Im K', '|K|']
keys   = [('K_re', 're_mean', 're_std'),
          ('K_im', 'im_mean', 'im_std'),
          ('abs_K','ab_mean','ab_std')]

for ax, lbl, (key, mkey, skey) in zip(axes, labels, keys):
    jitter = 0.0
    for s in spread_summary:
        xs = np.full_like(s[key], s['lam'], dtype=float)
        xs = xs + (np.arange(len(xs)) - (len(xs)-1)/2) * 0.002 * max(1e-3, s['lam'])
        ax.errorbar(xs, s[key], yerr=s['K_std'], fmt='o',
                    color='steelblue', alpha=0.65, ms=6, capsize=3, lw=1,
                    label='seed runs' if s is spread_summary[0] else None)
        best_i = list(s['seeds']).index(s['best_seed'])
        ax.plot(xs[best_i], s[key][best_i], marker='*', color='gold',
                ms=16, mec='black', mew=0.7, zorder=5,
                label='best seed' if s is spread_summary[0] else None)

    means = np.array([s[mkey] for s in spread_summary])
    stds  = np.array([s[skey] for s in spread_summary])
    ax.plot(lams_plot, means, '-', color='crimson', lw=1.3, alpha=0.9,
            label='across-seed mean')
    ax.fill_between(lams_plot, means - stds, means + stds,
                    color='crimson', alpha=0.15, label='mean \u00b1 std')
    ax.set_xlabel(r'$\lambda$')
    ax.set_ylabel(lbl)
    ax.set_title(f'{lbl}: per-seed runs, best = gold star')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='best')

plt.suptitle(
    f'11.1  Multi-seed spread  ({len(seed_list)} seeds per $\\lambda$)   '
    f'endpoints x_i={x_i_prop}, x_f={x_f_prop}',
    fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig("ml_multiseed_spread.png", dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved ml_multiseed_spread.png")

### 11.2 Monte-Carlo self-consistency (best model per $\lambda$)

Three internal checks on the **MC estimator itself** (the ML model is fixed; we just vary how we average it):

1. **$N_{\text{mc}}$ scaling.** Evaluate $K$ at $N_{\text{mc}} \in \{10\text{k},\,30\text{k},\,100\text{k},\,300\text{k}\}$. The **mean** should stabilise and the standard error should fall roughly as $1/\sqrt{N_{\text{mc}}}$. A log-log fit of $K_{\text{std}}$ vs $N_{\text{mc}}$ is plotted against the reference slope $-1/2$.

2. **Independent-batch check.** Draw 5 fresh batches of 100k samples with different random seeds and compute $K$ for each. The **empirical std across batches** should agree with the **single-batch** $K_{\text{std}}$ (within a small factor).

3. **Bootstrap CI.** Resample the stored 20k log-weights **with replacement** 500 times and compute $K$ on each resample. The 95% percentile interval gives a non-parametric confidence interval that does not assume Gaussian errors. It should be broadly consistent with the quoted $K \pm K_{\text{std}}$.

If $K$ drifts with $N_{\text{mc}}$, or the batch spread is much larger than $K_{\text{std}}$, or the bootstrap CI is asymmetric / much wider than $K_{\text{std}}$ — those are all signs that a few heavy paths are dominating.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Simple toggle: set back to True to re-enable Section 11.2.
run_mc_selfconsistency = False
 
n_mc_list     = [10_000, 30_000, 100_000, 300_000]
n_indep_batch = 5
batch_size_ib = 100_000
n_bootstrap   = 500

def eval_K_with_model(res, n_mc, rng_seed=None):
    """Re-run the MC estimator of K using the model stored in `res`,
    with fresh z samples.  Returns (K_complex, K_std)."""
    model      = res['model']
    lam        = res['lam']
    x_i, x_f   = res['x_i'], res['x_f']
    D_val      = res['D']
    m_val      = res['m']; omega_val = res['omega']; eps_val = res['eps']
    a_dt       = res['a_dt']
    norm_fac   = res['norm_factor_complex']

    if rng_seed is not None:
        torch.manual_seed(rng_seed)

    with torch.no_grad():
        z  = torch.randn(n_mc, D_val)
        x, a1, a2 = model(z)
        S  = complex_action(x, x_i, x_f, lam, a_dt, m_val, omega_val, eps_val)
        logP = -0.5 * torch.sum(z**2, dim=1) - (D_val / 2) * np.log(2 * np.pi)
        logdet = model.compute_log_det_J(z, a1, a2)
        logW = 1j * S + logdet - logP.to(torch.complex64)
        lw_max = torch.max(logW.real)
        W_shift = torch.exp(logW - lw_max)
        K  = norm_fac * torch.exp(lw_max).item() * torch.mean(W_shift).numpy()
        W_err = norm_fac * torch.exp(lw_max).item() * W_shift.numpy()
        K_std = np.std(W_err) / np.sqrt(n_mc)
    return complex(K), float(K_std)

def bootstrap_K(log_W, norm_factor, n_boot=500, rng_seed=0):
    """Bootstrap confidence interval on K from stored log-weights."""
    rng = np.random.default_rng(rng_seed)
    n = len(log_W)
    K_arr = np.empty(n_boot, dtype=np.complex128)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        lw  = log_W[idx]
        lw_max = np.max(lw.real)
        W = np.exp(lw - lw_max)
        K_arr[b] = norm_factor * np.exp(lw_max) * np.mean(W)
    return K_arr

if run_mc_selfconsistency:
    lams_for_val = sorted({r['lam'] for r in best_results})

    fig, axes = plt.subplots(len(lams_for_val), 2,
                             figsize=(13, 4.2 * len(lams_for_val)),
                             squeeze=False)

    print("=" * 100)
    print("11.2  MC SELF-CONSISTENCY  (N_mc scaling + independent batches + bootstrap)")
    print("=" * 100)

    for row, lam_val in enumerate(lams_for_val):
        res = [r for r in best_results if abs(r['lam'] - lam_val) < 1e-12][0]
        K_quoted    = res['K_re'] + 1j * res['K_im']
        K_std_quoted = res['std_err']

        print(f"\n--- lambda = {lam_val}  (best seed = {res['seed']}) ---")
        print(f"    Originally quoted: K = {K_quoted:.8f}   K_std = {K_std_quoted:.2e}")

        print(f"\n  (1) N_mc scaling:")
        print(f"    {'N_mc':>8}  {'Re K':>14}  {'Im K':>14}  {'K_std':>10}  "
              f"{'rel diff to quoted':>18}")
        scaling_Nmc, scaling_std, scaling_re, scaling_im = [], [], [], []
        for i, n_mc in enumerate(n_mc_list):
            K_i, Ks_i = eval_K_with_model(res, n_mc, rng_seed=1000 + i)
            rel = abs(K_i - K_quoted) / max(abs(K_quoted), 1e-30)
            print(f"    {n_mc:8d}  {K_i.real:+14.8f}  {K_i.imag:+14.8f}  "
                  f"{Ks_i:10.2e}  {rel:18.2e}")
            scaling_Nmc.append(n_mc); scaling_std.append(Ks_i)
            scaling_re.append(K_i.real); scaling_im.append(K_i.imag)
        scaling_Nmc = np.array(scaling_Nmc); scaling_std = np.array(scaling_std)

        ref_std = scaling_std[0] * np.sqrt(scaling_Nmc[0]) / np.sqrt(scaling_Nmc)
        slope, intercept = np.polyfit(np.log(scaling_Nmc), np.log(scaling_std), 1)
        print(f"    log-log fit slope = {slope:+.3f}  (ideal -0.500)")

        print(f"\n  (2) Independent-batch check "
              f"({n_indep_batch} x {batch_size_ib:,d} samples):")
        K_batches = []
        for b in range(n_indep_batch):
            K_b, Ks_b = eval_K_with_model(res, batch_size_ib, rng_seed=5000 + b)
            K_batches.append(K_b)
            print(f"    batch {b+1}: K = {K_b.real:+.8f} {K_b.imag:+.8f}j   "
                  f"(single-batch std = {Ks_b:.2e})")
        K_batches = np.array(K_batches)
        batch_std_re = K_batches.real.std(ddof=1)
        batch_std_im = K_batches.imag.std(ddof=1)
        ratio_re_b = batch_std_re / max(K_std_quoted, 1e-30)
        ratio_im_b = batch_std_im / max(K_std_quoted, 1e-30)
        print(f"    across-batch std(Re) = {batch_std_re:.2e}   "
              f"(ratio vs single-batch K_std = {ratio_re_b:.2f})")
        print(f"    across-batch std(Im) = {batch_std_im:.2e}   "
              f"(ratio vs single-batch K_std = {ratio_im_b:.2f})")
        batch_verdict = "OK" if max(ratio_re_b, ratio_im_b) < 3 else "SUSPICIOUS"
        print(f"    verdict: {batch_verdict}")

        print(f"\n  (3) Bootstrap CI from 20k stored log-weights, {n_bootstrap} resamples:")
        K_boot = bootstrap_K(res['log_W_sample'], res['norm_factor_complex'],
                             n_boot=n_bootstrap, rng_seed=7000 + row)
        re_lo, re_hi = np.percentile(K_boot.real, [2.5, 97.5])
        im_lo, im_hi = np.percentile(K_boot.imag, [2.5, 97.5])
        re_boot_std = K_boot.real.std(ddof=1)
        im_boot_std = K_boot.imag.std(ddof=1)
        print(f"    Re K: 95% CI = [{re_lo:+.8f}, {re_hi:+.8f}]   "
              f"bootstrap std = {re_boot_std:.2e}")
        print(f"    Im K: 95% CI = [{im_lo:+.8f}, {im_hi:+.8f}]   "
              f"bootstrap std = {im_boot_std:.2e}")
        print(f"    (Quoted K_std from 100k eval = {K_std_quoted:.2e}; "
              f"bootstrap uses only 20k samples so expect ~sqrt(5) wider.)")

        ax = axes[row, 0]
        ax.loglog(scaling_Nmc, scaling_std, 'o-', color='steelblue',
                  label='measured $K_{\\mathrm{std}}$')
        ax.loglog(scaling_Nmc, ref_std, '--', color='grey',
                  label=r'ideal $\propto 1/\sqrt{N_{\mathrm{mc}}}$')
        ax.set_xlabel(r'$N_{\mathrm{mc}}$')
        ax.set_ylabel(r'$K_{\mathrm{std}}$')
        ax.set_title(f'lam = {lam_val}: MC error vs $N_{{\\mathrm{{mc}}}}$  '
                     f'(fit slope = {slope:+.2f})')
        ax.grid(True, which='both', alpha=0.3)
        ax.legend(fontsize=9)

        ax = axes[row, 1]
        ax.scatter(K_boot.real, K_boot.imag, s=6, alpha=0.25,
                   color='steelblue', label='bootstrap samples')
        ax.scatter(K_batches.real, K_batches.imag, s=70, color='green',
                   marker='s', edgecolor='black', label='independent 100k batches')
        ax.scatter([K_quoted.real], [K_quoted.imag], s=150, color='crimson',
                   marker='*', edgecolor='black', label='quoted K (best seed)')
        ax.set_xlabel('Re K'); ax.set_ylabel('Im K')
        ax.set_title(f'lam = {lam_val}: bootstrap cloud + independent batches')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, loc='best')

    plt.suptitle('11.2  MC self-consistency per lambda', fontsize=14, y=1.0)
    plt.tight_layout()
    fig.savefig("ml_mc_selfconsistency.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("\nSaved ml_mc_selfconsistency.png")
else:
    print("11.2  MC self-consistency skipped. Set run_mc_selfconsistency = True to re-enable.")

### 11.3 Honest weight diagnostics

`ESS/N` near $1$ is necessary but not sufficient: a pathological sampler can still have high ESS if a small fraction of samples drive both the numerator and the denominator. For each $\lambda$ we therefore report **three numbers together** and **two distributions**.

**Numbers** (computed on the stored 20k log-weights):

- **ESS/N** = $|\langle W\rangle|^{2} / \langle|W|^{2}\rangle$ — aligned arrows vs cancelling ones.
- **Var($\log|W|$)** — spread of arrow **magnitudes** on a log scale. Should be small (a single scale).
- **max($|W|$) / mean($|W|$)** — concentration of magnitude. If this is big (say $\gtrsim 50$), a handful of samples dominate.

**Distributions** (one column per $\lambda$):

- **Histogram of $\log|W|$**. Healthy: narrow, **unimodal**, roughly symmetric. Red flag: long tail or multiple bumps.
- **Histogram of $\arg W$** (on $[-\pi, \pi]$). Healthy: narrow and unimodal near $0$. Red flag: **strong bimodality** — two clumps of phase $\approx \pm\pi/2$ or at $0$ and $\pi$ means the sum is close to a difference of two big numbers and the result is very sensitive to exact cancellation.

These diagnostics are deliberately **independent of the true answer**: they just test whether the estimator is statistically well-behaved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lams_for_diag = sorted({r['lam'] for r in best_results})
n_lam = len(lams_for_diag)

fig, axes = plt.subplots(n_lam, 2, figsize=(12, 3.2 * n_lam), squeeze=False)

print("=" * 100)
print("11.3  HONEST WEIGHT DIAGNOSTICS  (best seed per lambda, 20k stored log-weights)")
print("=" * 100)
print(f"  {'lam':>8}  {'seed':>6}  {'ESS/N':>8}  {'Var(log|W|)':>14}  "
      f"{'max|W|/mean|W|':>16}  {'median arg W':>15}  verdict")
print("  " + "-" * 98)

diag_summary = []
for row, lam_val in enumerate(lams_for_diag):
    res  = [r for r in best_results if abs(r['lam'] - lam_val) < 1e-12][0]
    logW = res['log_W_sample']

    lw_max    = np.max(logW.real)
    W_shifted = np.exp(logW - lw_max)
    log_absW  = logW.real - lw_max
    absW      = np.abs(W_shifted)
    argW      = np.angle(W_shifted)

    ess      = float(np.abs(np.mean(W_shifted))**2 / np.mean(np.abs(W_shifted)**2))
    var_logW = float(np.var(log_absW))
    max_rel  = float(absW.max() / absW.mean())
    arg_med  = float(np.median(argW))
    arg_std  = float(np.std(argW))

    red_flags = []
    if ess      < 0.5 : red_flags.append("low ESS")
    if var_logW > 1.0 : red_flags.append("wide |W|")
    if max_rel  > 50  : red_flags.append("one path dominates")
    if arg_std  > 0.8 : red_flags.append("wide phase")
    verdict = "HEALTHY" if not red_flags else "FLAGGED: " + ", ".join(red_flags)

    print(f"  {lam_val:8.4f}  {res['seed']:6d}  {ess:8.4f}  {var_logW:14.4e}  "
          f"{max_rel:16.2f}  {arg_med:+15.4f}  {verdict}")

    diag_summary.append(dict(
        lam=lam_val, seed=res['seed'],
        ess=ess, var_logW=var_logW, max_rel=max_rel,
        arg_median=arg_med, arg_std=arg_std, verdict=verdict,
    ))

    ax = axes[row, 0]
    ax.hist(log_absW, bins=80, color='steelblue', edgecolor='k',
            lw=0.3, alpha=0.85, density=True)
    ax.axvline(0, color='crimson', ls='--', lw=1,
               label=r'$\log|W|_{\max}$')
    ax.axvline(np.mean(log_absW), color='black', ls=':', lw=1,
               label=f'mean = {np.mean(log_absW):+.2f}')
    ax.set_xlabel(r'$\log|W| - \log|W|_{\max}$')
    ax.set_ylabel('density')
    ax.set_title(f'lam = {lam_val}   Var = {var_logW:.3f}   '
                 f'max/mean = {max_rel:.1f}   {"HEALTHY" if not red_flags else "FLAG"}')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[row, 1]
    ax.hist(argW, bins=np.linspace(-np.pi, np.pi, 81),
            color='seagreen', edgecolor='k', lw=0.3, alpha=0.85, density=True)
    ax.axvline(arg_med, color='crimson', ls='--', lw=1,
               label=f'median = {arg_med:+.3f}')
    ax.set_xlim(-np.pi, np.pi)
    ax.set_xlabel(r'$\arg W$  (radians)')
    ax.set_ylabel('density')
    ax.set_title(f'lam = {lam_val}   ESS/N = {ess:.3f}   '
                 f'phase std = {arg_std:.3f}')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle('11.3  Weight diagnostics per lambda  (log|W| and arg W)',
             fontsize=14, y=1.0)
plt.tight_layout()
fig.savefig("ml_weight_diagnostics.png", dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved ml_weight_diagnostics.png")

print("\n" + "=" * 100)
print("FINAL ML-ONLY VALIDATION SUMMARY")
print("=" * 100)
n_healthy = sum(1 for d in diag_summary if d['verdict'] == "HEALTHY")
print(f"  {n_healthy}/{len(diag_summary)} lambda values pass all weight-diagnostic checks.")
for d in diag_summary:
    print(f"    lam = {d['lam']:<8}  seed = {d['seed']:<6}  "
          f"ESS/N = {d['ess']:.3f}   Var(log|W|) = {d['var_logW']:.2e}   "
          f"max/mean = {d['max_rel']:6.1f}   -> {d['verdict']}")

### 11.4 Cancellation scatter: $|\Delta\theta|$ vs $|W|$

The histograms in 11.3 look at magnitudes and phases **separately**. A dangerous sample is one that is **both** long **and** misaligned — a long arrow pointing the wrong way contributes heavily to phase cancellation. To see these samples directly, plot each MC weight in the 2-D plane

$$\Big(\,\log|W|,\;\; |\Delta\theta|\,\Big)\qquad\text{where}\qquad \Delta\theta_i = \mathrm{wrap}\!\big(\arg W_i - \arg\langle W\rangle\big)\;\in[-\pi,\pi]$$

The reference direction $\arg\langle W\rangle$ is the **direction of the complex mean**, computed with angular wrapping (not the arithmetic mean of phases, which is broken across $\pm\pi$).

**How to read the scatter**:

- **bottom-right region** ($|W|$ large, $|\Delta\theta|$ small): good. These are the heavy contributors that all point together.
- **top-right region** ($|W|$ large, $|\Delta\theta|$ near $\pi/2$ or $\pi$): **dangerous**. Long arrows misaligned with the bulk — they drive the cancellation transverse to the mean.
- **top-left region** ($|W|$ tiny, $|\Delta\theta|$ large): harmless — misaligned but negligible weight.

We also report two scalar transverse-cancellation metrics:

- $\langle |W|\sin|\Delta\theta|\rangle / \langle|W|\rangle$ — fraction of weight perpendicular to the mean direction.
- $\langle |W|\,|\Delta\theta|\rangle / \langle|W|\rangle$ — weighted average angular distance.

Small is good, large is bad.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

lams_for_scatter = sorted({r['lam'] for r in best_results})
n_lam = len(lams_for_scatter)

n_cols = min(3, n_lam)
n_rows = math.ceil(n_lam / n_cols)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5.2 * n_cols, 4.2 * n_rows),
                         squeeze=False)

print("=" * 100)
print("11.4  |DELTA THETA| vs |W|  SCATTER  (best seed per lambda, 20k stored samples)")
print("=" * 100)
print(f"  {'lam':>8}  {'seed':>6}  {'mean_dir':>10}  "
      f"{'<|W| |sin dth|>/<|W|>':>22}  {'<|W| |dth|>/<|W|>':>20}")
print("  " + "-" * 98)

scatter_summary = []

for idx, lam_val in enumerate(lams_for_scatter):
    r = idx // n_cols
    c = idx % n_cols
    ax = axes[r, c]

    res  = [rr for rr in best_results if abs(rr['lam'] - lam_val) < 1e-12][0]
    logW = res['log_W_sample']

    # shift for numerical stability; shift cancels in phase / ratios below
    lw_max    = np.max(logW.real)
    W_shifted = np.exp(logW - lw_max)
    abs_W     = np.abs(W_shifted)

    # mean direction from the complex mean (not arithmetic mean of phases)
    W_mean     = np.mean(W_shifted)
    mean_angle = float(np.angle(W_mean))

    # wrapped angular distance in [-pi, pi]
    raw      = np.angle(W_shifted) - mean_angle
    d_theta  = (raw + np.pi) % (2 * np.pi) - np.pi
    abs_dth  = np.abs(d_theta)

    # transverse-cancellation metrics (shift factor cancels in ratios)
    frac_perp = float(np.mean(abs_W * np.abs(np.sin(d_theta))) / max(np.mean(abs_W), 1e-300))
    frac_dth  = float(np.mean(abs_W * abs_dth)                 / max(np.mean(abs_W), 1e-300))

    print(f"  {lam_val:8.4f}  {res['seed']:6d}  {mean_angle:+10.4f}  "
          f"{frac_perp:22.4e}  {frac_dth:20.4e}")
    scatter_summary.append(dict(
        lam=lam_val, seed=res['seed'], mean_angle=mean_angle,
        frac_perp=frac_perp, frac_dth=frac_dth,
    ))

    log_abs_rel = np.log10(abs_W / abs_W.max() + 1e-300)

    sc = ax.scatter(log_abs_rel, abs_dth,
                    c=abs_W * abs_dth,
                    cmap='magma', s=3, alpha=0.35,
                    norm=plt.Normalize(vmin=0,
                                       vmax=np.percentile(abs_W * abs_dth, 99)))
    ax.axhline(np.pi / 2, color='orange', ls='--', lw=0.8, alpha=0.7,
               label=r'$|\Delta\theta| = \pi/2$')
    ax.axhline(np.pi,     color='crimson', ls='--', lw=0.8, alpha=0.7,
               label=r'$|\Delta\theta| = \pi$ (opposite)')
    ax.set_xlabel(r'$\log_{10}\!\left(|W|/|W|_{\max}\right)$')
    ax.set_ylabel(r'$|\Delta\theta|$  (radians)')
    ax.set_ylim(0, np.pi)
    ax.set_title(
        f'lam = {lam_val}   seed = {res["seed"]}\n'
        f"$\\langle |W|\\,|\\sin\\Delta\\theta|\\rangle / \\langle|W|\\rangle$ = {frac_perp:.2e}",
        fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc='upper left')
    cbar = plt.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label(r'$|W|\,|\Delta\theta|$', fontsize=8)

# blank out unused axes
for k in range(n_lam, n_rows * n_cols):
    r = k // n_cols
    c = k % n_cols
    axes[r, c].axis('off')

plt.suptitle(
    r'11.4  Cancellation scatter: $|\Delta\theta|$ vs $|W|$   '
    r'(bottom-right = heavy & aligned = good;   top-right = heavy & misaligned = dangerous)',
    fontsize=12, y=1.0)
plt.tight_layout()
fig.savefig("ml_cancellation_scatter.png", dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved ml_cancellation_scatter.png")

print("\n" + "=" * 100)
print("11.4  SUMMARY of transverse-cancellation metrics")
print("=" * 100)
for d in scatter_summary:
    tag = "clean" if d['frac_perp'] < 0.1 else ("some" if d['frac_perp'] < 0.5 else "HEAVY")
    print(f"    lam = {d['lam']:<8}  seed = {d['seed']:<6}  "
          f"transverse fraction = {d['frac_perp']:.3e}   ({tag} cancellation)")

In [ ]:
for r in all_results:
    print(r['lam'], r['seed'], r['K_re'] + 1j*r['K_im'], r['final_loss'], r['ess_n'])